# Pipeline klasyfikatora gestów dłoni

**Autorzy:** Maciej Bik, Kamil Buszta, Wiktor Cioch, Arkadiusz Cios

---

Klasyfikator **SVM** (Support Vector Machine) rozpoznaje 10 gestów dłoni. SVM jest modelem matematycznym, który szuka hiperpłaszczyzny rozdzielającej klasy w przestrzeni wielowymiarowej — nie przetworzy bezpośrednio surowego zdjęcia. Potrzebuje **wektora liczb o stałej, identycznej długości** dla każdego przykładu.

Pipeline preprocesingu przekształca dowolne zdjęcie dłoni w znormalizowany wektor **12 288 wartości**. Każda z tych wartości opisuje jeden fragment informacji wizualnej: czy w tym miejscu jest skóra, czy jest krawędź, jak wygląda gradient.

Każde zdjęcie przechodzi przez 9 kroków. Na końcu trzy mapy 64×64 px są ułożone jako kanały tensora i spłaszczane do wektora wejściowego SVM.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from method.method import extract_features, _skin_mask, _saliency_u8, _hog_image
from scripts.loaders import NUSIIDatasetLoader

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 9

def bgr2rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def show_pair(title, i1, i2, cmap='gray', vmin=0, vmax=255, l1='Obraz 1', l2='Obraz 2'):
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
    fig.suptitle(title, fontsize=12, fontweight='bold')
    fig.subplots_adjust(top=0.76)
    kw = dict(cmap=cmap, vmin=vmin, vmax=vmax) if cmap else {}
    ax[0].imshow(i1, **kw); ax[0].set_title(l1, fontsize=9, pad=12); ax[0].axis('off')
    ax[1].imshow(i2, **kw); ax[1].set_title(l2, fontsize=9, pad=12); ax[1].axis('off')
    plt.tight_layout(); plt.show()

print('OK')

In [ ]:
DATASET_PATH = os.path.join('..', 'NUS-Hand-Posture-Dataset-II', 'Hand Postures')
files = NUSIIDatasetLoader.get_learning_files(base_path=DATASET_PATH, limit=60, shuffle=False)
print(f'Znaleziono {len(files)} obrazow')

img1_path, label1 = files[1]
img2_path, label2 = files[30]
img1_bgr = cv2.imread(img1_path)
img2_bgr = cv2.imread(img2_path)

print(f'Obraz 1: {os.path.basename(img1_path)}, gest {label1}, {img1_bgr.shape[1]}x{img1_bgr.shape[0]}')
print(f'Obraz 2: {os.path.basename(img2_path)}, gest {label2}, {img2_bgr.shape[1]}x{img2_bgr.shape[0]}')

In [ ]:
show_pair('Oryginalne obrazy wejsciowe', bgr2rgb(img1_bgr), bgr2rgb(img2_bgr), cmap=None,
          l1=f'Obraz 1 — gest {label1} ({img1_bgr.shape[1]}x{img1_bgr.shape[0]} px)',
          l2=f'Obraz 2 — gest {label2} ({img2_bgr.shape[1]}x{img2_bgr.shape[0]} px)')

---
## Krok 0 — Skalowanie do 64×64 px

**Dlaczego?**
SVM szuka granicy decyzyjnej w przestrzeni o ustalonej liczbie wymiarów — jeśli jeden obraz ma wektor 10 000 elementów, a inny 25 000, nie da się ich porównać ani nauczyć jednego modelu. Wszystkie obrazy muszą mieć dokładnie taki sam rozmiar wektora. Rozmiar 64×64 to wynik kompromisu: wystarczająco mały, żeby trenowanie SVM na tysiącach obrazów było szybkie, wystarczająco duży, żeby zachować kształt palców i wcięcia między nimi — kluczowe cechy rozróżniające gesty.

**Jak działa?** Interpolacja dwuliniowa — każdy nowy piksel jest ważoną średnią czterech sąsiednich pikseli z oryginalnego obrazu. Zapewnia płynne przeskalowanie bez schodkowatości.

**Wejście:** dowolny rozmiar, 3 kanały BGR, uint8, wartości 0–255

**Wyjście:** 64×64 px, 3 kanały BGR, uint8, wartości 0–255

In [ ]:
img1 = cv2.resize(img1_bgr, (64, 64))
img2 = cv2.resize(img2_bgr, (64, 64))

fig, axes = plt.subplots(2, 2, figsize=(8, 7))
fig.suptitle('Krok 0: Skalowanie do 64x64', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.78)
axes[0,0].imshow(bgr2rgb(img1_bgr)); axes[0,0].set_title(f'Obraz 1 — oryginalny\n({img1_bgr.shape[1]}x{img1_bgr.shape[0]} px)', pad=12); axes[0,0].axis('off')
axes[0,1].imshow(bgr2rgb(img1));     axes[0,1].set_title('Obraz 1 — po skalowaniu\n(64x64 px)', pad=12);                          axes[0,1].axis('off')
axes[1,0].imshow(bgr2rgb(img2_bgr)); axes[1,0].set_title(f'Obraz 2 — oryginalny\n({img2_bgr.shape[1]}x{img2_bgr.shape[0]} px)', pad=12); axes[1,0].axis('off')
axes[1,1].imshow(bgr2rgb(img2));     axes[1,1].set_title('Obraz 2 — po skalowaniu\n(64x64 px)', pad=12);                          axes[1,1].axis('off')
plt.tight_layout(); plt.show()

---
## Krok 1 — F1: Skala szarości

**Dlaczego?**
Ten sam gest wykonany przez osobę o ciemnej i jasnej karnacji powinien dawać te same cechy — kolor skóry nie niesie informacji o kształcie. Poza tym obraz kolorowy ma trzy kanały, a operacje Canny i HOG działają na jednym kanale szarości. Skala szarości upraszcza dane bez utraty informacji o geometrii.

Dodatkowy efekt: piksel RGB zmienia wszystkie trzy wartości przy zmianie oświetlenia, natomiast gradient jasności (zmiana wartości między sąsiednimi pikselami) jest dużo bardziej stabilny. Dalsze operacje (Canny, HOG) bazują właśnie na gradientach — więc szarość to właściwe wejście.

**Jak działa?** Trzy kanały BGR łączone są ważoną sumą: Y = 0.114·B + 0.587·G + 0.299·R. Wagi odzwierciedlają czułość ludzkiego oka — zielony wydaje nam się najjaśniejszy.

**Wejście:** 64×64, 3 kanały BGR, uint8, 0–255

**Wyjście:** 64×64, **1 kanał**, uint8, zakres **0–255**
- 0 = czarny (ciemny obszar)
- 255 = biały (jasny obszar)
- wartości pośrednie = odcienie szarości

> F1 jest wejściem do Canny i HOG. **Nie wchodzi** bezpośrednio do tensoru dla SVM.

In [ ]:
F1_1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
F1_2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.8))
fig.suptitle('Krok 1: Skala szarosci — 1 kanal, wartosci 0-255', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.76)
axes[0].imshow(bgr2rgb(img1))
axes[0].set_title('Wejscie: obraz 64x64\n(wynik kroku 0 — skalowanie)\n3 kanaly BGR', pad=12); axes[0].axis('off')
axes[1].imshow(F1_1, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Wyjscie — Obraz 1\nskala szarosci\n(1 kanal, wartosci 0-255)', pad=12); axes[1].axis('off')
axes[2].imshow(F1_2, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('Wyjscie — Obraz 2\nskala szarosci\n(1 kanal, wartosci 0-255)', pad=12); axes[2].axis('off')
plt.tight_layout(); plt.show()

---
## Krok 2 — Maska koloru skóry (F_SC)

**Dlaczego?**
Tło obrazu zawiera masę zbędnych informacji — meble, ściany, ubrania — które generują krawędzie i gradienty niezwiązane z gestem. Zamiast analizować cały obraz, chcemy skupić się wyłącznie na obszarze dłoni. Kolor skóry to naturalna wskazówka, gdzie dłoń się znajduje.

Przestrzeń HSV zamiast RGB, bo w RGB ten sam kolor skóry wygląda zupełnie różnie w zależności od oświetlenia — ciemne pomieszczenie zmienia wartości R, G, B znacząco. W HSV kanał **H (odcień)** koduje kolor niezależnie od jasności. Skóra zawsze mieści się w zakresie H=0–17 (odcienie ciepłe, pomarańczowo-żółte), niezależnie od tego czy jest jaśniej czy ciemniej.

**Jak działa?** Konwersja BGR→HSV, następnie funkcja cv2.inRange zaznacza piksel jako skórę gdy: H∈[0,17] (ciepłe odcienie), S∈[15,170] (wyklucza biel i bardzo intensywne kolory), V∈[0,255] (dowolna jasność).

**Wejście:** 64×64, 3 kanały BGR, uint8, 0–255

**Wyjście:** 64×64, **1 kanał**, uint8, **wyłącznie wartości 0 lub 255** — maska binarna
- 255 = ten piksel wygląda jak skóra
- 0 = ten piksel nie wygląda jak skóra
- **nie ma wartości pośrednich**

In [ ]:
skin1 = _skin_mask(img1)
skin2 = _skin_mask(img2)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle('Krok 2: Maska koloru skory — BINARNA (tylko 0 lub 255)', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.78)

for row, (img, sk) in enumerate([(img1, skin1), (img2, skin2)]):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    pct = (sk > 0).sum() / sk.size * 100
    axes[row,0].imshow(bgr2rgb(img))
    axes[row,0].set_title('Wejscie: obraz 64x64\n(wynik kroku 0 — skalowanie)\n3 kanaly BGR', pad=12); axes[row,0].axis('off')
    axes[row,1].imshow(hsv[:,:,0], cmap='hsv', vmin=0, vmax=180)
    axes[row,1].set_title('Kanal H (odcien barwy) po konwersji\nBGR → HSV. Cieple odcienie\n(H=0-17) = kolor skory', pad=12); axes[row,1].axis('off')
    axes[row,2].imshow(sk, cmap='gray', vmin=0, vmax=255)
    axes[row,2].set_title(f'Wyjscie: MASKA SKORY\nBialy = skora ({pct:.0f}% pikseli)\nCzarny = nie skora | TYLKO 0 lub 255', pad=12); axes[row,2].axis('off')

plt.tight_layout(); plt.show()

---
## Krok 3 — Mapa saliencji (F_S)

**Dlaczego?**
Maska skóry ma jeden poważny problem: beżowe ściany, drewniane biurka, czy inne części ciała mogą mieć podobną barwę do skóry i zostać błędnie zaznaczone. Potrzebujemy drugiego, niezależnego kryterium potwierdzającego, że coś jest dłonią.

Saliencja odpowiada na pytanie: *co w tym obrazie wyróżnia się na tle otoczenia?* Dłoń umieszczona przed jednolitym tłem zawsze będzie najbardziej wizualnie wyróżniającym się obiektem. Kombinacja AND (krok 6) z maską skóry eliminuje fałszywe detekcje: obiekt musi być jednocześnie w kolorze skóry ORAZ wyróżniać się wizualnie.

**Jak działa?** Algorytm StaticSaliencyFineGrained analizuje lokalne kontrasty kolorystyczne i jasności w całym obrazie w przestrzeni częstotliwości. Każdy piksel dostaje wartość proporcjonalną do tego, jak bardzo różni się od otoczenia. Wynik float32 z zakresu 0.0–1.0 jest skalowany do 0–255.

**Wejście:** 64×64, 3 kanały BGR, uint8, 0–255

**Wyjście:** 64×64, **1 kanał**, uint8, zakres **0–255 z wartościami pośrednimi** — skala szarości
- 255 = obszar bardzo wyróżniający się
- 0 = obszar monotonny, zlewa się z otoczeniem
- wartości pośrednie (np. 80, 150, 200) = stopniowe wyróżnienie

In [ ]:
sal1 = _saliency_u8(img1)
sal2 = _saliency_u8(img2)

fig, axes = plt.subplots(2, 2, figsize=(9, 8))
fig.suptitle('Krok 3: Mapa saliencji — 1 kanal, szarosc 0-255', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.78)

for row, (img, sal) in enumerate([(img1, sal1), (img2, sal2)]):
    axes[row,0].imshow(bgr2rgb(img))
    axes[row,0].set_title('Wejscie: obraz 64x64\n(wynik kroku 0 — skalowanie)\n3 kanaly BGR', pad=12); axes[row,0].axis('off')
    axes[row,1].imshow(sal, cmap='gray', vmin=0, vmax=255)
    axes[row,1].set_title(
        'Wyjscie: MAPA SALIENCJI\n'
        'Bialy = obszar wyrozniony (prawdopodobnie dlon)\n'
        'Czarny = tlo, monotonne otoczenie',
        pad=12); axes[row,1].axis('off')

plt.tight_layout(); plt.show()

---
## Krok 4 — Krawędzie Canny'ego

**Dlaczego?**
Kształt gestu jest zakodowany przede wszystkim w konturze dłoni — ile palców widać, jak są rozłożone, czy są złączone czy rozstawione. Krawędzie bezpośrednio opisują te kontury. Są też odporne na zmianę jasności: przesuniecie oświetlenia nie tworzy nowych krawędzi ani nie usuwa istniejących — ważne są zmiany jasności, a nie jej absolutna wartość.

Podwójne progowanie (50/150) usuwa szum przy jednoczesnym zachowaniu prawdziwych krawędzi: krawędź musi być albo wystarczająco silna (>150), albo sąsiadować z silną krawędzią. Dzięki temu losowy szum pikseli nie generuje fałszywych krawędzi.

**Jak działa?** Rozmycie Gaussem (usunięcie szumu), gradienty Sobela (siła i kierunek zmiany jasności), Non-Maximum Suppression (cienkie jednopiksele krawędzie), progowanie z histerezą — próg dolny 50, próg górny 150.

**Wejście:** F1 — 64×64, 1 kanał, szarość 0–255

**Wyjście:** 64×64, **1 kanał**, uint8, **wyłącznie wartości 0 lub 255** — maska binarna
- 255 = wykryto krawędź w tym pikselu
- 0 = brak krawędzi
- **nie ma wartości pośrednich**

In [ ]:
canny1 = cv2.Canny(F1_1, 50, 150)
canny2 = cv2.Canny(F1_2, 50, 150)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
fig.suptitle('Krok 4: Krawedzie Canniego — BINARNE (tylko 0 lub 255)', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.78)

for row, (img, f1, can) in enumerate([(img1, F1_1, canny1), (img2, F1_2, canny2)]):
    overlay = bgr2rgb(img).copy()
    overlay[can > 0] = [255, 60, 60]
    axes[row,0].imshow(f1, cmap='gray', vmin=0, vmax=255)
    axes[row,0].set_title('Wejscie: skala szarosci\n(wynik kroku 1)\nszukamy gwaltownych zmian jasnosci', pad=12); axes[row,0].axis('off')
    axes[row,1].imshow(can, cmap='gray', vmin=0, vmax=255)
    axes[row,1].set_title('Wyjscie: WYKRYTE KRAWEDZIE\nBialy = jest krawedz tutaj\nCzarny = nie ma krawedzi | TYLKO 0 lub 255', pad=12); axes[row,1].axis('off')
    axes[row,2].imshow(overlay)
    axes[row,2].set_title('Krawedzie naniesione na oryginal\n(czerwone piksele = miejsce\nwykrytej krawedzi)', pad=12); axes[row,2].axis('off')

plt.tight_layout(); plt.show()

---
## Krok 5 — Wizualizacja HOG (Histogram of Oriented Gradients)

**Dlaczego?**
Canny daje binarne informacje — krawędź jest albo jej nie ma. Traci przy tym dwie ważne informacje: jak silna jest ta krawędź i w jakim kierunku biegnie. HOG te informacje zachowuje.

Dla rozpoznawania gestów kierunek gradientu jest istotny: wyciągnięty palec pionowy tworzy głównie gradienty poziome (po bokach palca), a palec poziomy — głównie gradienty pionowe. Canny nie rozróżni tych dwóch przypadków jeśli kontur jest podobnie długi, ale HOG — tak. Normalizacja bloków 2×2 w HOG sprawia też, że cechy są odporne na małe przesunięcia dłoni w kadrze.

**Jak działa?** Obraz dzielony jest na komórki 8×8 pikseli. W każdej komórce obliczany jest histogram kierunków gradientów w 9 kubełkach (co 20°, od 0° do 180°). Wynik normalizowany w blokach 2×2 komórek i wizualizowany jako obraz: jasność piksela = siła gradientu w tym miejscu. Parametry wg Dalal & Triggs CVPR 2005.

**Wejście:** F1 — 64×64, 1 kanał, szarość 0–255

**Wyjście:** 64×64, **1 kanał**, uint8, zakres **0–255 z wartościami pośrednimi** — skala szarości
- 255 = silny, wyraźny gradient
- 0 = brak gradientu, jednolity obszar
- wartości pośrednie = różna siła gradientu

In [ ]:
hog1 = _hog_image(img1)
hog2 = _hog_image(img2)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
fig.suptitle('Krok 5: Gradienty HOG — SZAROSC (wartosci 0-255, nie binarne)', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.78)

for row, (f1, can, h) in enumerate([(F1_1, canny1, hog1), (F1_2, canny2, hog2)]):
    axes[row,0].imshow(f1, cmap='gray', vmin=0, vmax=255)
    axes[row,0].set_title('Wejscie: skala szarosci\n(wynik kroku 1)', pad=12); axes[row,0].axis('off')
    axes[row,1].imshow(h, cmap='gray', vmin=0, vmax=255)
    axes[row,1].set_title('Wyjscie: GRADIENTY HOG\nenergia i kierunek zmian jasnosci\nJasny = silny gradient | Ciemny = brak', pad=12); axes[row,1].axis('off')
    axes[row,2].imshow(can, cmap='gray', vmin=0, vmax=255)
    axes[row,2].set_title('Canny dla porownania\n(wynik kroku 4, binarny 0/255)\nvs HOG powyzej (wartosci 0-255)', pad=12); axes[row,2].axis('off')

plt.tight_layout(); plt.show()

---
## Krok 6 — F2 = Maska skóry AND Saliencja

**Dlaczego?**
Maska skóry wykrywa kolor — może się mylić w tle. Saliencja wykrywa kontrast wizualny — może reagować na inne wyróżniające się obiekty w kadrze (jasna lampa, odzież w kontrastowym kolorze). Żadna z nich z osobna nie jest wystarczająco niezawodna.

AND wymaga spełnienia **obu warunków jednocześnie**. Cokolwiek ma kolor skóry, ale nie wyróżnia się wizualnie (np. ściana z podobnym odcieniem) — wypada. Cokolwiek się wyróżnia wizualnie, ale nie ma koloru skóry — wypada. Zostają tylko te piksele, które spełniają oba kryteria niezależnie od siebie, co znacząco redukuje fałszywe detekcje.

**Jak działa?** Bitowy AND: każdy bit wyjściowy = 1 tylko gdy odpowiadające bity obu operandów = 1. Ponieważ F_SC jest binarna (0 lub 255=11111111): tam gdzie F_SC=0, wynik=0; tam gdzie F_SC=255, wynik=wartość saliencji.

**Wejście:** F_SC (binarna: 0 lub 255) i F_S (szarość: 0–255)

**Wyjście:** 64×64, 1 kanał, uint8, zakres **0–255**
- Jasny = obszar skóry potwierdzony przez saliencję
- Ciemny = poza skórą lub obszar niewyróżniający się

In [ ]:
F2_1 = cv2.bitwise_and(skin1, sal1)
F2_2 = cv2.bitwise_and(skin2, sal2)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle('Krok 6: Obszar dloni = Maska skory AND Mapa saliencji', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.78)

for row, (sk, sal, f2) in enumerate([(skin1, sal1, F2_1), (skin2, sal2, F2_2)]):
    axes[row,0].imshow(sk, cmap='gray', vmin=0, vmax=255)
    axes[row,0].set_title('Wejscie 1: Maska skory\n(wynik kroku 2, binarna: 0 lub 255)\nBialy = ten piksel to skora', pad=12); axes[row,0].axis('off')
    axes[row,1].imshow(sal, cmap='gray', vmin=0, vmax=255)
    axes[row,1].set_title('Wejscie 2: Mapa saliencji\n(wynik kroku 3, szarosc: 0-255)\nJasny = wyrozniony obszar', pad=12); axes[row,1].axis('off')
    axes[row,2].imshow(f2, cmap='gray', vmin=0, vmax=255)
    axes[row,2].set_title('Wyjscie: maska AND saliencja\nZostaje tylko gdzie OBA sa > 0\n(skora i wyrozniony obszar)', pad=12); axes[row,2].axis('off')

plt.tight_layout(); plt.show()

---
## Krok 7 — F3 = Canny OR HOG

**Dlaczego?**
Canny i HOG to dwa komplementarne detektory struktury obrazu. Canny jest precyzyjny — daje cienkie, jednopiksele krawędzie na konturze palców. HOG jest szerszy — wykrywa gradienty też wewnątrz dłoni (linie stawów, zmiany faktury skóry) i zachowuje informację o kierunku.

OR łączy je bez utraty czegokolwiek. Użycie AND tutaj byłoby błędem — wymagałoby żeby oba detektory zgadzały się co do każdego piksela, a ponieważ Canny daje cienkie linie a HOG grubsze obszary, ich AND byłby prawie pusty. OR daje pełny obraz struktury: tam gdzie Canny widzi krawędź, tam F3=255; tam gdzie tylko HOG widzi gradient, tam F3=wartość HOG.

**Jak działa?** Bitowy OR: każdy bit wyjściowy = 1 gdy przynajmniej jeden z wejściowych bitów = 1. Ponieważ Canny jest binarny (0 lub 255=11111111): tam gdzie Canny=255, wynik=255; tam gdzie Canny=0, wynik=wartość HOG.

**Wejście:** Canny (binarna: 0 lub 255) i HOG (szarość: 0–255)

**Wyjście:** 64×64, 1 kanał, uint8, zakres **0–255**
- 255 = krawędź wykryta przez Canny
- wartość HOG = gradient bez krawędzi Canny
- 0 = brak krawędzi i brak gradientu

In [ ]:
F3_1 = cv2.bitwise_or(canny1, hog1)
F3_2 = cv2.bitwise_or(canny2, hog2)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle('Krok 7: Pelna mapa krawedzi = Canny OR HOG', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.78)

for row, (can, h, f3) in enumerate([(canny1, hog1, F3_1), (canny2, hog2, F3_2)]):
    axes[row,0].imshow(can, cmap='gray', vmin=0, vmax=255)
    axes[row,0].set_title('Wejscie 1: Krawedzie Canny\n(wynik kroku 4, binarna: 0 lub 255)\nBialy = krawedz', pad=12); axes[row,0].axis('off')
    axes[row,1].imshow(h, cmap='gray', vmin=0, vmax=255)
    axes[row,1].set_title('Wejscie 2: Gradienty HOG\n(wynik kroku 5, szarosc: 0-255)\nJasny = silny gradient', pad=12); axes[row,1].axis('off')
    axes[row,2].imshow(f3, cmap='gray', vmin=0, vmax=255)
    axes[row,2].set_title('Wyjscie: Canny OR HOG\nAktywny gdy PRZYNAJMNIEJ\nJEDNO z nich jest niezerowe', pad=12); axes[row,2].axis('off')

plt.tight_layout(); plt.show()

---
## Krok 8 — F4 = (F2 AND F3) XOR Maska skóry

**Dlaczego?**
Potrzebujemy mapy opisującej bryłę dłoni — nie jej kontury (to F3), nie sam obszar koloru skóry (to F2), ale kształt wypełnienia, czyli jak wygląda wnętrze dłoni. Wnętrze różni się między gestami: zamknięta pięść ma duże jednolite wnętrze, rozłożona dłoń ma wąskie wnętrza między palcami.

**Jak działa?** Dwa etapy:

**Etap A — pośredni = F2 AND F3:** Szukamy pikseli, które są jednocześnie w obszarze dłoni (F2 > 0) i na krawędzi lub gradiencie (F3 > 0). To krawędzie leżące wewnątrz obszaru skóry.

**Etap B — F4 = pośredni XOR skin:** XOR daje 1 gdy bity są różne. Ponieważ skin jest binarna (0 lub 255=11111111): poza skórą XOR z 0 nie zmienia nic; wewnątrz skóry XOR z 255 odwraca wszystkie bity. Piksele z silnymi krawędziami (wysoki pośredni ≈ 255) po odwróceniu dają niski F4. Piksele bez krawędzi (pośredni ≈ 0) po odwróceniu dają wysoki F4.

Wynik: F4 jest jasny w środku dłoni (brak krawędzi), ciemny na konturach palców (krawędzie).

**Wejście:** F2 (0–255), F3 (0–255), F_SC (0 lub 255)

**Wyjście:** 64×64, 1 kanał, uint8, zakres **0–255**
- Jasny = wnętrze dłoni (skóra bez krawędzi)
- Ciemny = kontury palców lub obszar poza dłonią

In [ ]:
inter1 = cv2.bitwise_and(F2_1, F3_1)
inter2 = cv2.bitwise_and(F2_2, F3_2)
F4_1 = cv2.bitwise_xor(inter1, skin1)
F4_2 = cv2.bitwise_xor(inter2, skin2)

fig, axes = plt.subplots(2, 5, figsize=(17, 8))
fig.suptitle('Krok 8: Sylwetka dloni = (obszar dloni AND krawedzie) XOR maska skory', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.78)

for row, (f2, f3, inter, sk, f4) in enumerate([(F2_1,F3_1,inter1,skin1,F4_1),(F2_2,F3_2,inter2,skin2,F4_2)]):
    for col, (data, ttl) in enumerate([
        (f2,    'Obszar dloni\n(skora + saliencja)'),
        (f3,    'Mapa krawedzi\n(Canny + HOG)'),
        (inter, 'Posredni = obszar AND krawedzie\n(krawedzie wewnatrz obszaru dloni)'),
        (sk,    'Maska skory\n(binarna: 0 lub 255)'),
        (f4,    'SYLWETKA DLONI\n(wnętrze bez krawedzi\n= jasne tam gdzie brak krawedzi)')
    ]):
        axes[row,col].imshow(data, cmap='gray', vmin=0, vmax=255)
        axes[row,col].set_title(ttl, fontsize=8, pad=12); axes[row,col].axis('off')

plt.tight_layout(); plt.show()

---
## Krok 9 — Złożenie F2, F3, F4 → wejście SVM

**Dlaczego trzy mapy, a nie jedna?**
Każda mapa opisuje inny aspekt dłoni:
- **F2** mówi *gdzie jest dłoń* (obszar skóry potwierdzony saliencją)
- **F3** mówi *jak wygląda kontur* (krawędzie i gradienty)
- **F4** mówi *jak wygląda wnętrze* (bryła dłoni bez konturów)

SVM uczący się na wszystkich trzech jednocześnie może skombinować te trzy perspektywy — np. gest "pięść" ma małe F4 (mało wnętrza), a gest "stop" ma duże F4 i charakterystyczny kontur F3.

**Jak wygląda wejście do SVM?**
Trzy mapy 64×64 układane są jako trzy kanały tensora 64×64×3 (jak obraz RGB, ale to nie są kanały kolorów — to trzy różne mapy cech). Tensor jest spłaszczany do jednowymiarowego wektora 12 288 liczb. SVM widzi tylko ten wektor — nie wie, że to były obrazy.

**Jak działa?** np.stack([F2, F3, F4], axis=-1) tworzy tensor 64×64×3. .flatten() zamienia go na wektor 12 288 elementów.

**Wejście:** F2, F3, F4 — każda 64×64, uint8, 0–255

**Wyjście:** wektor **12 288 wartości** (64 × 64 × 3 = 12 288)

In [ ]:
final1 = np.stack([F2_1, F3_1, F4_1], axis=-1)
final2 = np.stack([F2_2, F3_2, F4_2], axis=-1)

fig, axes = plt.subplots(2, 4, figsize=(14, 8))
fig.suptitle('Krok 9: Trzy mapy szarosci — kazda opisuje inny aspekt dloni', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.78)

for row, (f2, f3, f4) in enumerate([(F2_1,F3_1,F4_1),(F2_2,F3_2,F4_2)]):
    axes[row,0].imshow(f2, cmap='gray', vmin=0, vmax=255)
    axes[row,0].set_title('GDZIE JEST DLON (F2)\nszarosc 0-255\n(skora + saliencja)', pad=12); axes[row,0].axis('off')
    axes[row,1].imshow(f3, cmap='gray', vmin=0, vmax=255)
    axes[row,1].set_title('KONTURY DLONI (F3)\nszarosc 0-255\n(krawedzie + gradienty)', pad=12); axes[row,1].axis('off')
    axes[row,2].imshow(f4, cmap='gray', vmin=0, vmax=255)
    axes[row,2].set_title('WNETRZE DLONI (F4)\nszarosc 0-255\n(sylwetka)', pad=12); axes[row,2].axis('off')
    axes[row,3].imshow(np.stack([f2,f3,f4], axis=-1))
    axes[row,3].set_title('Trzy mapy zlozone razem\n(kolor SZTUCZNY — to nie RGB!\nto tylko wizualizacja)', pad=12); axes[row,3].axis('off')

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Co SVM faktycznie widzi — wektor 12 288 liczb', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.76)

vec1 = final1.flatten().astype(float)

axes[0].imshow(vec1.reshape(1, -1), cmap='gray', vmin=0, vmax=255, aspect='auto')
axes[0].set_title('Wektor 12 288 wartosci wyswietlony jako poziomy pasek\nLewa czesc = F2 (gdzie dlon) | Srodek = F3 (krawedzie) | Prawa = F4 (sylwetka)', pad=12)
axes[0].set_xlabel('Indeks elementu (0 → 12287)')
axes[0].set_yticks([])

axes[1].hist(vec1, bins=50, color='steelblue', edgecolor='k', alpha=0.8)
axes[1].set_title('Rozklad wartosci w tym wektorze\n(ile pikseli ma dana jasnosc)', pad=12)
axes[1].set_xlabel('Wartosc piksela (0 = czarny, 255 = bialy)')
axes[1].set_ylabel('Liczba elementow')
axes[1].axvline(vec1.mean(), color='red', linestyle='--', label=f'srednia = {vec1.mean():.0f}')
axes[1].legend()

plt.tight_layout(); plt.show()
print(f'Ksztalt tensora przed spłaszczeniem: {final1.shape}')
print(f'Po flatten(): wektor {final1.flatten().shape[0]} elementow — to jest wejscie do SVM')

---
## Wszystkie etapy — widok zbiorczy dla obu obrazów

In [ ]:
def full_pipeline_view(img_bgr, label):
    img   = cv2.resize(img_bgr, (64,64))
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sk    = _skin_mask(img)
    sal   = _saliency_u8(img)
    can   = cv2.Canny(gray, 50, 150)
    h     = _hog_image(img)
    f2    = cv2.bitwise_and(sk, sal)
    f3    = cv2.bitwise_or(can, h)
    f4    = cv2.bitwise_xor(cv2.bitwise_and(f2,f3), sk)

    stages = [
        (bgr2rgb(img), '0. Resize 64x64\noryginalny kolorowy',          None),
        (gray,         '1. Skala szarosci\n1 kanal, 0-255',             'gray'),
        (sk,           '2. Maska skory\nbinarna: 0 lub 255',            'gray'),
        (sal,          '3. Mapa saliencji\nszarosc: 0-255',             'hot'),
        (can,          '4. Krawedzie Canny\nbinarne: 0 lub 255',        'gray'),
        (h,            '5. Gradienty HOG\nszarosc: 0-255',              'gray'),
        (f2,           '6. Gdzie jest dlon\n(skora AND saliencja)',      'gray'),
        (f3,           '7. Kontury dloni\n(Canny OR HOG)',              'gray'),
        (f4,           '8. Sylwetka dloni\n(posredni XOR skora)',       'gray'),
        (np.stack([f2,f3,f4],axis=-1), '9. Wejscie SVM\n(3 mapy zlozone)', None),
    ]
    fig, axes = plt.subplots(2, 5, figsize=(18, 8))
    fig.suptitle(f'Pelny pipeline — gest {label}', fontsize=13, fontweight='bold')
    fig.subplots_adjust(top=0.78)
    for ax, (data, title, cmap) in zip(axes.flatten(), stages):
        kw = {'cmap': cmap, 'vmin': 0, 'vmax': 255} if cmap else {}
        ax.imshow(data, **kw); ax.set_title(title, fontsize=8, pad=10); ax.axis('off')
    plt.tight_layout(); plt.show()

full_pipeline_view(img1_bgr, label1)
full_pipeline_view(img2_bgr, label2)

---
---
# Przykład 6×6 — operacje bitowe na rzeczywistych wartościach

Poniżej pipeline odtworzony na planszy **6×6 pikseli** z wartościami odpowiadającymi rzeczywistym typom danych używanym w kodzie.

---

### Typy danych czterech map wejściowych

| Mapa | Typ | Zakres wartości | Dlaczego taki format |
|------|-----|-----------------|---------------------|
| **F_SC** (maska skóry) | binarna | **tylko 0 lub 255** | cv2.inRange zwraca tylko 0/255 |
| **F_S** (saliencja) | skala szarości | **0–255 z wartościami pośrednimi** | algorytm saliencji zwraca float skalowany do uint8 |
| **Canny** | binarna | **tylko 0 lub 255** | detektor krawędzi zwraca tylko 0/255 |
| **HOG** | skala szarości | **0–255 z wartościami pośrednimi** | energia gradientu normalizowana do zakresu 0–255 |

Na planszy środkowy blok (wiersze 1–4, kolumny 1–4) = dłoń.

In [ ]:
skin6 = np.array([
    [  0,   0,   0,   0,   0,   0],
    [  0, 255, 255, 255, 255,   0],
    [  0, 255, 255, 255, 255,   0],
    [  0, 255, 255, 255, 255,   0],
    [  0, 255, 255, 255, 255,   0],
    [  0,   0,   0,   0,   0,   0],
], dtype=np.uint8)

sal6 = np.array([
    [  0,   0,   0,   0,   0,   0],
    [  0, 210, 180, 140,  20,   0],
    [  0, 190, 220,  30,  10,   0],
    [  0,  15, 170, 200,  25,   0],
    [  0,  10, 155, 230, 200,   0],
    [  0,   0,   0,   0,   0,   0],
], dtype=np.uint8)

canny6 = np.array([
    [  0,   0,   0,   0,   0,   0],
    [  0, 255,   0,   0, 255,   0],
    [  0, 255,   0,   0, 255,   0],
    [  0, 255,   0,   0, 255,   0],
    [  0, 255,   0,   0, 255,   0],
    [  0,   0,   0,   0,   0,   0],
], dtype=np.uint8)

hog6 = np.array([
    [  0,   0,   0,   0,   0,   0],
    [  0,  30, 180, 200,  25,   0],
    [  0,  20, 210, 195,  15,   0],
    [  0,  35, 175, 220,  30,   0],
    [  0,  25, 190, 210,  20,   0],
    [  0,   0,   0,   0,   0,   0],
], dtype=np.uint8)

F2_6   = skin6 & sal6
F3_6   = canny6 | hog6
inter6 = F2_6 & F3_6
F4_6   = inter6 ^ skin6

print('Macierze obliczone. Wartosci F4 (po XOR ze skora):')
print(F4_6)

In [ ]:
def show_mat6(ax, mat, title, binary=False):
    if binary:
        cmap_b = ListedColormap(['#1a1a2e', '#f0a500'])
        ax.imshow(mat, cmap=cmap_b, vmin=0, vmax=255)
    else:
        ax.imshow(mat, cmap='gray', vmin=0, vmax=255)
    for i in range(6):
        for j in range(6):
            v = int(mat[i,j])
            bright = (v > 120) if binary else (v > 160)
            ax.text(j, i, str(v), ha='center', va='center',
                    fontsize=9, fontweight='bold',
                    color='#111' if bright else 'white')
    ax.set_title(title, fontsize=9, fontweight='bold', pad=10)
    ax.set_xticks(np.arange(-0.5,6,1), minor=True)
    ax.set_yticks(np.arange(-0.5,6,1), minor=True)
    ax.grid(which='minor', color='#555', lw=0.7)
    ax.tick_params(which='both', bottom=False, left=False, labelbottom=False, labelleft=False)

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Dane wejsciowe 6x6 — rzeczywiste wartosci pikseli', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.80, bottom=0.05)

show_mat6(axes[0], skin6,  'F_SC — maska skory\nBINARNA: tylko 0 lub 255', binary=True)
show_mat6(axes[1], sal6,   'F_S — saliencja\nSZAROSC: 0-255 (wartosci posrednie)')
show_mat6(axes[2], canny6, 'Canny — krawedzie\nBINARNE: tylko 0 lub 255', binary=True)
show_mat6(axes[3], hog6,   'HOG — gradienty\nSZAROSC: 0-255 (wartosci posrednie)')
plt.show()

---
### F2 = F_SC AND F_S

F_SC jest binarna — zawiera tylko 0 i 255 (binarnie: 00000000 i 11111111). Operacja AND z taką wartością ma prostą właściwość:

- **0 AND cokolwiek = 0** — zero ma wszystkie bity równe 0, więc wynik zawsze 0
- **255 AND cokolwiek = to cokolwiek** — 255 ma wszystkie bity równe 1, więc niczego nie zmienia

Efekt: F2 jest identyczny z F_S (saliencja) wewnątrz obszaru skóry, a zero poza nim. Wartości pośrednie saliencji (140, 30, 170...) zostają bez zmian.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('F2 = F_SC AND F_S', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.80, bottom=0.05)

show_mat6(axes[0], skin6, 'F_SC\n(binarna: 0 lub 255)', binary=True)
show_mat6(axes[1], sal6,  'F_S\n(szarosc: 0-255)')
show_mat6(axes[2], F2_6,  'F2 = F_SC AND F_S\n(wynik: 0-255)')
axes[3].axis('off')
axes[3].set_facecolor('#1a1a2e')
txt = (
    'Operacja AND — 3 przypadki:\n\n'
    'skin=  0 (00000000)\n'
    'sal = 210 (11010010)\n'
    'AND ─────────────────\n'
    'F2  =   0 (00000000)\n'
    '  skin=0 wycina wartosc sal\n\n'
    'skin=255 (11111111)\n'
    'sal = 180 (10110100)\n'
    'AND ─────────────────\n'
    'F2  = 180 (10110100)\n'
    '  skin=255 zachowuje sal\n\n'
    'skin=255 (11111111)\n'
    'sal =  20 (00010100)\n'
    'AND ─────────────────\n'
    'F2  =  20 (00010100)\n'
    '  mala sal. rowniez zachowana'
)
axes[3].text(0.05, 0.95, txt, transform=axes[3].transAxes,
             va='top', ha='left', fontsize=9, family='monospace',
             color='white', linespacing=1.6)
plt.show()

---
### F3 = Canny OR HOG

Canny jest binarna — zawiera tylko 0 i 255. Operacja OR z taką wartością:

- **255 OR cokolwiek = 255** — 255 ma wszystkie bity równe 1, więc wynik zawsze 255
- **0 OR cokolwiek = to cokolwiek** — zero nie zmienia żadnego bitu

Efekt: F3 ma wartość 255 wszędzie gdzie Canny wykrył krawędź, a wartość HOG tam gdzie Canny nic nie wykrył. Wartości pośrednie HOG (180, 200, 175...) są zachowane bez zmian.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('F3 = Canny OR HOG', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.80, bottom=0.05)

show_mat6(axes[0], canny6, 'Canny\n(binarna: 0 lub 255)', binary=True)
show_mat6(axes[1], hog6,   'HOG\n(szarosc: 0-255)')
show_mat6(axes[2], F3_6,   'F3 = Canny OR HOG\n(wynik: 0-255)')
axes[3].axis('off')
axes[3].set_facecolor('#1a1a2e')
txt = (
    'Operacja OR — 3 przypadki:\n\n'
    'canny=255 (11111111)\n'
    'hog  =  30 (00011110)\n'
    ' OR ─────────────────\n'
    'F3   = 255 (11111111)\n'
    '  canny=255 nadpisuje HOG\n\n'
    'canny=  0 (00000000)\n'
    'hog  =180 (10110100)\n'
    ' OR ─────────────────\n'
    'F3   =180 (10110100)\n'
    '  canny=0 przepuszcza HOG\n\n'
    'canny=  0 (00000000)\n'
    'hog  =200 (11001000)\n'
    ' OR ─────────────────\n'
    'F3   =200 (11001000)\n'
    '  canny=0 przepuszcza HOG'
)
axes[3].text(0.05, 0.95, txt, transform=axes[3].transAxes,
             va='top', ha='left', fontsize=9, family='monospace',
             color='white', linespacing=1.6)
plt.show()

---
### F4 = (F2 AND F3) XOR F_SC — dwa etapy

**Etap A — pośredni = F2 AND F3**

Tu oba operandy mają wartości 0–255 — nie ma skrótu jak poprzednio. AND działa bit po bicie między dwiema wartościami pośrednimi. Wynik zależy od nakładania się bitów obu operandów.

**Etap B — F4 = pośredni XOR F_SC**

XOR: bit wynikowy = 1 gdy bity wejściowe są różne. Ponieważ F_SC jest binarna:
- **x XOR 0 = x** — poza skórą: wartość nie zmienia się
- **x XOR 255 = NOT(x)** — wewnątrz skóry: wszystkie bity odwrócone

Konsekwencja: im wyższy był pośredni (silna krawędź), tym niższy F4. Piksele bez krawędzi (pośredni ≈ 0) dają po XOR z 255 wysoką wartość F4 ≈ 255.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 10))
fig.suptitle('F4 = (F2 AND F3) XOR F_SC — etap A i B', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.88, hspace=0.35)

show_mat6(axes[0,0], F2_6,   'F2\n(wejscie etapu A)')
show_mat6(axes[0,1], F3_6,   'F3\n(wejscie etapu A)')
show_mat6(axes[0,2], inter6, 'POSREDNI = F2 AND F3\n(etap A — wynik)')
axes[0,3].axis('off'); axes[0,3].set_facecolor('#1a1a2e')
txt_a = (
    'AND dwoch wartosci 0-255\n(bity moga sie nie pokrywac):\n\n'
    'F2[1,2]=180 (10110100)\n'
    'F3[1,2]=180 (10110100)\n'
    'AND ─────────────────\n'
    'pos  = 180 (10110100)\n\n'
    'F2[2,3]= 30 (00011110)\n'
    'F3[2,3]=195 (11000011)\n'
    'AND ─────────────────\n'
    'pos  =   2 (00000010)\n'
    '  bity sie nie pokrywaja!\n\n'
    'F2[3,3]=200 (11001000)\n'
    'F3[3,3]=220 (11011100)\n'
    'AND ─────────────────\n'
    'pos  = 200 (11001000)'
)
axes[0,3].text(0.05, 0.95, txt_a, transform=axes[0,3].transAxes,
               va='top', ha='left', fontsize=9, family='monospace',
               color='white', linespacing=1.6)

show_mat6(axes[1,0], inter6, 'POSREDNI\n(wejscie etapu B)')
show_mat6(axes[1,1], skin6,  'F_SC\n(maska XOR)', binary=True)
show_mat6(axes[1,2], F4_6,   'F4 = posredni XOR F_SC\n(etap B — wynik)')
axes[1,3].axis('off'); axes[1,3].set_facecolor('#1a1a2e')
txt_b = (
    'XOR z binarna maska\n(skin=255 odwraca bity):\n\n'
    'pos[1,1]=210 (11010010)\n'
    'skin =  255 (11111111)\n'
    'XOR ─────────────────\n'
    'F4  =  45 (00101101)\n'
    '  wysoki pos. → niski F4\n\n'
    'pos[1,4]= 20 (00010100)\n'
    'skin =  255 (11111111)\n'
    'XOR ─────────────────\n'
    'F4  = 235 (11101011)\n'
    '  niski pos. → wysoki F4\n\n'
    'pos[2,3]=  2 (00000010)\n'
    'skin =  255 (11111111)\n'
    'XOR ─────────────────\n'
    'F4  = 253 (11111101)\n'
    '  prawie zero → prawie 255'
)
axes[1,3].text(0.05, 0.95, txt_b, transform=axes[1,3].transAxes,
               va='top', ha='left', fontsize=9, family='monospace',
               color='white', linespacing=1.6)
plt.show()

---
### Podsumowanie przykładu 6×6 — wszystkie mapy

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 10))
fig.suptitle('Pelny pipeline 6x6 — wejscia i wyjscia kazdego kroku', fontsize=12, fontweight='bold')
fig.subplots_adjust(top=0.88, hspace=0.35)

show_mat6(axes[0,0], skin6,  'F_SC — maska skory\nBINARNA: 0 lub 255', binary=True)
show_mat6(axes[0,1], sal6,   'F_S — saliencja\nSZAROSC: 0-255')
show_mat6(axes[0,2], canny6, 'Canny — krawedzie\nBINARNE: 0 lub 255', binary=True)
show_mat6(axes[0,3], hog6,   'HOG — gradienty\nSZAROSC: 0-255')

show_mat6(axes[1,0], F2_6,   'F2 = F_SC AND F_S\n(0-255)')
show_mat6(axes[1,1], F3_6,   'F3 = Canny OR HOG\n(0-255)')
show_mat6(axes[1,2], inter6, 'posredni = F2 AND F3\n(0-255)')
show_mat6(axes[1,3], F4_6,   'F4 = posredni XOR F_SC\n(0-255)')
plt.show()

---
## Tabela podsumowująca — typy danych na każdym etapie

| Krok | Operacja | Wejście | Format wyjścia |
|------|----------|---------|----------------|
| 0 | Resize 64×64 | dowolny rozmiar BGR | uint8, 0–255, 3 kanały |
| 1 | BGR → skala szarości | 64×64 BGR | uint8, **0–255**, 1 kanał |
| 2 | BGR → HSV + progowanie | 64×64 BGR | uint8, **tylko 0 lub 255**, 1 kanał |
| 3 | Saliencja StaticSaliencyFineGrained | 64×64 BGR | uint8, **0–255 z wartościami pośrednimi**, 1 kanał |
| 4 | Canny(gray, 50, 150) | F1 szarość | uint8, **tylko 0 lub 255**, 1 kanał |
| 5 | HOG + wizualizacja | F1 szarość | uint8, **0–255 z wartościami pośrednimi**, 1 kanał |
| 6 | bitwise_and(F_SC, F_S) | F_SC + F_S | uint8, 0–255, 1 kanał |
| 7 | bitwise_or(Canny, HOG) | Canny + HOG | uint8, 0–255, 1 kanał |
| 8 | bitwise_xor(F2∧F3, F_SC) | F2+F3+F_SC | uint8, 0–255, 1 kanał |
| 9 | np.stack([F2,F3,F4]) + flatten() | F2, F3, F4 | wektor **12 288 wartości** → SVM |